# Reservoir MIDI

Reservoir MIDI learns the notes of a MIDI file with a quantum reservoir and generates a new arrangement as a fresh MIDI file. Train once, then generate as many takes as you like from the trained model. (The API names it QRC MIDI.)

## Setup

In [ ]:
import getpass
import os

from moth_atlas import Atlas

ENGINE = "qrc-midi-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "qrc-midi-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import pretty_midi
from IPython.display import Audio, display


def upload(path):
    """A MIDI file as (name, bytes, type) for `files`: Python can't guess the type of a .mid file, and the engine
    only accepts audio/midi."""
    return (os.path.basename(path), open(path, "rb").read(), "audio/midi")


def midi(data):
    """A MIDI file (path or bytes) as a PrettyMIDI object."""
    return pretty_midi.PrettyMIDI(data if isinstance(data, str) else io.BytesIO(data))


def rolls(clips, play=True, seconds=20):
    """Draw {label: MIDI path or bytes} as piano rolls, one row each, then play each (sine voices, the first `seconds`)."""
    fig, axes = plt.subplots(len(clips), 1, figsize=(9, 1.8 * len(clips)), sharex=True, squeeze=False)
    for ax, (label, data) in zip(axes[:, 0], clips.items()):
        pm = midi(data)
        for i, inst in enumerate(pm.instruments):
            for n in inst.notes:
                ax.barh(n.pitch, n.end - n.start, left=n.start, height=0.8, color=f"C{i}", alpha=0.3 + 0.7 * n.velocity / 127)
        count = sum(len(inst.notes) for inst in pm.instruments)
        ax.set_ylabel(f"{label}\n{count} notes", rotation=0, ha="right", va="center")
    axes[-1, 0].set_xlabel("seconds")
    plt.show()
    if play:
        for label, data in clips.items():
            print(label)
            x = midi(data).synthesize(fs=22050)[: 22050 * seconds]
            display(Audio(0.9 * x / max(1e-9, np.abs(x).max()), rate=22050, normalize=False))


def output_asset(job, slot):
    """One output file of a finished job, as an Asset you can pass straight into another job's files."""
    out = next(o for o in job.outputs if o.slot == slot)
    return client.assets.get(out.output_asset_id)

## Try it

`phrase.mid` is a short original phrase beside this notebook (melody and bass, 100 bpm). The rolls are drawn from the files; the audio is a plain sine rendering of the notes.

In [ ]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "length": 32,
        "quality": "fast",
        "variation": 1,
        "bpm": 100,
        "seed": 7,
    },
    files={
        "midi": upload("phrase.mid"),
    },
)
rolls({"input": "phrase.mid", "generated": job.output_bytes("result")})

Keep the trained `model` and generate another take from it, without training again.

In [ ]:
model = output_asset(job, "model")

job = client.jobs.run(engine=ENGINE, params={"length": 32, "bpm": 100, "seed": 8}, files={"model": model})
rolls({"another take": job.output_bytes("result")})

## Parameters

What to expect from each:

- `length` (up to 500): how many notes to generate.
- `variation` (above 0): how freely the result departs from the learned order. Low is tight and repetitive, high loose and surprising.
- `quality`: how hard to train on a fresh file: `instant` (no training, a quick shuffle), `fast`, `moderate` (default) or `complete` (slow and tight).
- `seed`: fixes a result. On a fresh file it also fixes the learned reservoir; with a `model` it just picks a different take.
- `loop`: treat the melody as looping, so its end flows back into its start (learning only).
- `bpm`: the tempo written into the file. Note lengths are kept either way; this only sets the grid a DAW shows.
- `velocity` (1 to 127): how hard every generated note is struck.

Inputs: `midi` to learn from, and/or `model` to skip training. Outputs: `result` (the MIDI), `model` and `state`.

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Tokens.** The input is cut into a vocabulary of tokens: here, the notes of the file, and the piece becomes a sequence of them.

**Reservoir.** The sequence drives a quantum reservoir: a small, fixed quantum system whose state is nudged by each token in turn. Its measured features hold a fading memory of what came before, richer than the last token alone. The reservoir itself is never trained.

**Readout.** Only a simple readout is trained, mapping the reservoir's features to the next token. `quality` sets how hard it trains (`instant` skips training for a quick untrained shuffle). With `loop`, the end of the piece is treated as flowing back into its start.

**Generate.** Starting from the reservoir's state, the engine predicts the next token, samples one (`variation` sets how far it may stray from the most likely), feeds it back in and repeats `length` times.

The trained readout comes back as the `model` output: pass it into later runs to generate more takes without training again. The method follows *Level Generation with Quantum Reservoir Computing* (arXiv 2505.13287).

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Variation

Three takes from the same trained model: close to the phrase, then further from it.

In [ ]:
clips = {}
for variation in (0.2, 1.0, 3.0):
    print(f"variation={variation}")
    job = client.jobs.run(engine=ENGINE, params={"length": 32, "variation": variation, "bpm": 100, "seed": 1},
                          files={"model": model}, quiet=True)
    clips[f"variation {variation}"] = job.output_bytes("result")
rolls(clips, play=False)

### Quality

Fresh training each time: `instant` is an untrained shuffle, `complete` learns the phrase tightly.

In [ ]:
clips = {}
for quality in ("instant", "fast", "complete"):
    print(f"quality={quality}")
    job = client.jobs.run(engine=ENGINE, params={"length": 32, "quality": quality, "bpm": 100, "seed": 7},
                          files={"midi": upload("phrase.mid")}, quiet=True)
    clips[quality] = job.output_bytes("result")
rolls(clips, play=False)

### Loop

With `loop` on, the phrase's last note leads back to its first, so generation can cycle; off, the end is an end.

In [ ]:
clips = {}
for loop in (True, False):
    print(f"loop={loop}")
    job = client.jobs.run(engine=ENGINE, params={"length": 48, "quality": "fast", "loop": loop, "bpm": 100, "seed": 7},
                          files={"midi": upload("phrase.mid")}, quiet=True)
    clips[f"loop {'on' if loop else 'off'}"] = job.output_bytes("result")
rolls(clips, play=False)

## Questions

**What makes this quantum?**

The memory: the reservoir is a quantum system, and its measured features, which carry the melody's history, come from a quantum state much larger than its number of qubits suggests. Only the classical readout on top is trained.

**What happens to chords and several tracks?**

The notes are learned as one sequence. Simple melodies and lines give the clearest results.

**Why does the sample play as sine tones?**

The notebook renders the notes with a plain sine voice so you can hear them here. Open the MIDI in any DAW to hear it with real instruments.

**Is this running on quantum hardware?**

No, the reservoir is simulated.